# 🎯 Tool Calling 도전 과제 — A, D

- **과제 A**: 단위 변환기 (`ConversionResult` Pydantic 정형 반환)
- **과제 D**: docstring 짧게 vs 자세히 — LLM의 도구 선택 비교 실험

> 구성: **[기본 셋업]** 셀은 한 번만 실행 → **[실습]** 셀은 과제별로 독립 실행

# 🔧 기본 셋업

In [1]:
# 환경 설정 + API 키 로드
import os
from dotenv import load_dotenv

load_dotenv()
print("✅ API 키 OK" if os.environ.get("OPENAI_API_KEY") else "❌ 키 없음")

✅ API 키 OK


In [ ]:
# 공통 import + LLM
from typing import Literal
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, ToolMessage, AIMessage
from langchain_core.tools import tool

# Tool Calling은 temperature=0 권장 (도구 선택의 일관성)
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
print("✅ import 완료")

✅ import 완료


In [ ]:
# 헬퍼 함수: 질문 + 도구 → 4단계 흐름 실행 → 최종 답변
def run_with_tools(question: str, tools: list, llm_obj=llm):
    llm_with_tools = llm_obj.bind_tools(tools)
    messages: list[HumanMessage | AIMessage | ToolMessage] = [HumanMessage(content=question)]

    response = llm_with_tools.invoke(messages)
    messages.append(response)

    if not response.tool_calls:
        return response.content

    tool_map = {t.name: t for t in tools}
    for tc in response.tool_calls:
        result = tool_map[tc["name"]].invoke(tc["args"])
        messages.append(ToolMessage(content=str(result), tool_call_id=tc["id"]))

    return llm_with_tools.invoke(messages).content

print("✅ 헬퍼 함수 준비 완료")

✅ 헬퍼 함수 준비 완료


---

# 🧪 실습 A. 단위 변환기

길이·무게·온도 변환 도구 3개. 반환값은 모두 `ConversionResult` 객체.

## A-1. 스키마 + 도구 정의

In [39]:
# 정형 반환 스키마
class ConversionResult(BaseModel):
    original_value: float = Field(description="변환 전 값")
    original_unit: str = Field(description="변환 전 단위")
    converted_value: float = Field(description="변환 후 값")
    converted_unit: str = Field(description="변환 후 단위")

In [40]:
# 기준단위: 길이(m)
LENGTH_TO_M = {"m": 1.0, "cm": 0.01, "km": 1000.0, "inch": 0.0254, "ft": 0.3048, "mile": 1609.344}
LengthUnit = Literal["m", "cm", "km", "inch", "ft", "mile"]

@tool
def convert_length(value: float, from_unit: LengthUnit, to_unit: LengthUnit) -> ConversionResult:
    """길이 단위를 변환합니다.

    Args:
        value: 변환할 값
        from_unit: 출발 단위 ('m', 'cm', 'km', 'inch', 'ft', 'mile')
        to_unit: 도착 단위 ('m', 'cm', 'km', 'inch', 'ft', 'mile')
    """
    converted = value * LENGTH_TO_M[from_unit] / LENGTH_TO_M[to_unit] # 1m -> cm: 1 * 1 / 0.01 = 100cm
    return ConversionResult(original_value=value, original_unit=from_unit,
                            converted_value=converted, converted_unit=to_unit)

In [41]:
# 기준단위: 무게(kg)
WEIGHT_TO_KG = {"kg": 1.0, "g": 0.001, "lb": 0.45359237, "oz": 0.028349523125}
WeightUnit = Literal["kg", "g", "lb", "oz"]

@tool
def convert_weight(value: float, from_unit: WeightUnit, to_unit: WeightUnit) -> ConversionResult:
    """무게 단위를 변환합니다.

    Args:
        value: 변환할 값
        from_unit: 출발 단위 ('kg', 'g', 'lb', 'oz')
        to_unit: 도착 단위 ('kg', 'g', 'lb', 'oz')
    """
    converted = value * WEIGHT_TO_KG[from_unit] / WEIGHT_TO_KG[to_unit] # 1kg -> g: 1 * 1 / 0.001 = 1000g
    return ConversionResult(original_value=value, original_unit=from_unit,
                            converted_value=converted, converted_unit=to_unit)

In [42]:
# 온도는 배율이 아니라 공식이 필요 → 섭씨(C)를 기준으로 경유
_TO_C = {"C": lambda x: x, "F": lambda x: (x - 32) * 5 / 9, "K": lambda x: x - 273.15}
_FROM_C = {"C": lambda c: c, "F": lambda c: c * 9 / 5 + 32, "K": lambda c: c + 273.15}
TempUnit = Literal["C", "F", "K"]   # Literal → LLM이 없는 단위를 만들 수 없음

@tool
def convert_temperature(value: float, from_unit: TempUnit, to_unit: TempUnit) -> ConversionResult:
    """온도 단위를 변환합니다.

    Args:
        value: 변환할 값
        from_unit: 출발 단위 ('C' 섭씨, 'F' 화씨, 'K' 켈빈)
        to_unit: 도착 단위 ('C' 섭씨, 'F' 화씨, 'K' 켈빈)
    """
    converted = _FROM_C[to_unit](_TO_C[from_unit](value))
    return ConversionResult(original_value=value, original_unit=from_unit,
                            converted_value=converted, converted_unit=to_unit)

## A-2. LLM 연결 — 과제 테스트 질문

힌트의 두 질문 + 무게·온도 질문. LLM이 질문에 맞는 도구를 골라야 합니다.

In [44]:
QUERIES = [
      "180cm가 몇 미터야?",
      "10km는 몇 마일이야?",
      "70kg은 몇 파운드야?",
      "섭씨 36.5도는 화씨로 얼마야?"
]

for q in QUERIES:
    print(f"❓ {q}")
    print(f"💬 {run_with_tools(q, [convert_length, convert_weight, convert_temperature])}")
    print()

❓ 180cm가 몇 미터야?
💬 180cm는 1.8미터입니다.

❓ 10km는 몇 마일이야?
💬 10킬로미터는 약 6.21마일입니다.

❓ 70kg은 몇 파운드야?
💬 70kg은 약 154.32파운드입니다.

❓ 섭씨 36.5도는 화씨로 얼마야?
💬 섭씨 36.5도는 화씨로 97.7도입니다.



---

# 🧪 실습 D. docstring 일부러 짧게 써보기

같은 함수, docstring만 다른 두 도구. LLM은 **이름·docstring·타입 힌트**만 보고 도구를 고릅니다.

## D-1. 두 버전 정의

In [45]:
@tool
def add_v1(a: int, b: int) -> int:
    """더하기"""        # ← 너무 짧음
    return a + b

@tool
def add_v2(a: int, b: int) -> int:
    """
    두 정수의 합을 계산합니다.
    수학 계산이 필요한 모든 경우에 사용하세요.
    """
    return a + b

for i, t in enumerate([add_v1, add_v2]):
    print(f"{i+1}. {t.name}: {t.description!r}")

1. add_v1: '더하기'
2. add_v2: '두 정수의 합을 계산합니다.\n수학 계산이 필요한 모든 경우에 사용하세요.'


## D-2. 비교 함수

`tool_calls`(LLM이 도구를 부르려 했는지)만 보면 되므로 **2차 호출 없이** 1차 응답만 확인합니다.

In [47]:
def called_tools(question: str, tools: list) -> list[str]:
    """LLM이 호출하려는 도구 이름 목록 (빈 리스트 = 도구 안 씀)"""
    response = llm.bind_tools(tools).invoke(question)
    return [tc["name"] for tc in response.tool_calls]

QUERIES = [
    "3 더하기 5는?",                                # 명확한 덧셈
    "123456789 + 987654321 계산해줘",               # 큰 수 덧셈
    "사과 12개랑 배 7개를 합치면 과일이 몇 개야?",   # 일상어 표현
    "우리 팀 인원 4명에 신입 3명이 더 오면 몇 명이야?", # '덧셈'이라는 말 없음
    "파이썬에 대해 한 줄로 설명해줘.",                  # 도구 불필요 (대조군)
]

## D-3. 실험 ① 각 버전을 단독 등록

도구가 하나뿐이어도 docstring이 짧으면 안 부를까요?

In [ ]:
for i, q in enumerate(QUERIES):
    r1 = called_tools(q, [add_v1])
    r2 = called_tools(q, [add_v2])
    print(f"Q. {i+1}. {q}")
    print(f"   v1 (짧음)  : {r1 or '호출 안 함'}")
    print(f"   v2 (자세히): {r2 or '호출 안 함'}\n")

Q. 3 더하기 5는?
   v1 (짧음)  : ['add_v1']
   v2 (자세히): ['add_v2']

Q. 123456789 + 987654321 계산해줘
   v1 (짧음)  : ['add_v1']
   v2 (자세히): ['add_v2']

Q. 사과 12개랑 배 7개를 합치면 과일이 몇 개야?
   v1 (짧음)  : ['add_v1']
   v2 (자세히): ['add_v2']

Q. 우리 팀 인원 4명에 신입 3명이 더 오면 몇 명이야?
   v1 (짧음)  : ['add_v1']
   v2 (자세히): ['add_v2']

Q. 파이썬에 대해 한 줄로 설명해줘.
   v1 (짧음)  : 호출 안 함
   v2 (자세히): 호출 안 함



## D-4. 실험 ② 두 버전을 동시에 등록

경쟁 상황에서 LLM이 어느 쪽을 고르는지 봅니다.

In [ ]:
for i, q in enumerate(QUERIES):
    print(f"Q. {i+1}. {q}")
    print(f"  - 선택: {called_tools(q, [add_v1, add_v2]) or '호출 안 함'}\n")

Q. 3 더하기 5는?
  - 선택: ['add_v1']

Q. 123456789 + 987654321 계산해줘
  - 선택: ['add_v1', 'add_v2']

Q. 사과 12개랑 배 7개를 합치면 과일이 몇 개야?
  - 선택: ['add_v1', 'add_v2']

Q. 우리 팀 인원 4명에 신입 3명이 더 오면 몇 명이야?
  - 선택: ['add_v1']

Q. 파이썬에 대해 한 줄로 설명해줘.
  - 선택: 호출 안 함



## D-5. 실험 ③ 이름을 중립으로 — 경쟁 상황 + 등록 순서 교차

`add`라는 이름이 덧셈임을 알려주는 단서가 되므로, 이름을 `tool_a` / `tool_b`로 바꿔 **docstring만** 단서로 남깁니다.

- `tool_a`: 짧은 docstring, `tool_b`: 자세한 docstring (동작은 동일)
- **등록 순서를 `[a, b]` / `[b, a]`로 교차**해서, 선택이 docstring을 따르는지 위치를 따르는지 구분합니다.

In [49]:
@tool
def tool_a(a: int, b: int) -> int:
    """더하기"""        # ← 짧음
    return a + b

@tool
def tool_b(a: int, b: int) -> int:
    """
    두 정수의 합을 계산합니다.
    수학 계산이 필요한 모든 경우에 사용하세요.
    """
    return a + b

for t in [tool_a, tool_b]:
    print(f"{t.name}: {t.description!r}")

tool_a: '더하기'
tool_b: '두 정수의 합을 계산합니다.\n수학 계산이 필요한 모든 경우에 사용하세요.'


In [ ]:
orders = {"[a, b]": [tool_a, tool_b], "[b, a]": [tool_b, tool_a]}

for i, q in enumerate(QUERIES):
    print(f"Q. {i+1}. {q}")
    for label, tools in orders.items():
        print(f"  - 등록 순서 {label}: {called_tools(q, tools) or '호출 안 함'}")
    print()

Q. 3 더하기 5는?
  - 등록 순서 [a, b]: ['tool_a']
  - 등록 순서 [b, a]: ['tool_a']

Q. 123456789 + 987654321 계산해줘
  - 등록 순서 [a, b]: ['tool_b']
  - 등록 순서 [b, a]: ['tool_a']

Q. 사과 12개랑 배 7개를 합치면 과일이 몇 개야?
  - 등록 순서 [a, b]: ['tool_a', 'tool_b']
  - 등록 순서 [b, a]: ['tool_a']

Q. 우리 팀 인원 4명에 신입 3명이 더 오면 몇 명이야?
  - 등록 순서 [a, b]: ['tool_a']
  - 등록 순서 [b, a]: ['tool_a']

Q. 파이썬에 대해 한 줄로 설명해줘.
  - 등록 순서 [a, b]: 호출 안 함
  - 등록 순서 [b, a]: 호출 안 함



**역할이 같은 도구끼리 경쟁할 때는 docstring 길이가 선택에 큰 영향을 주지 않았다.**

## D-6. 실험 ④ 기능은 다른데 docstring이 모호한 경우

실험 ③은 두 도구가 **같은 일**을 해서 어느 쪽을 골라도 정답이었습니다. 이번에는 **기능이 다른** 도구(합 / 곱)로 바꿔 선택의 **정답 여부**를 채점합니다.

- **모호 버전**: 이름 `calc_a`(합), `calc_b`(곱), docstring은 둘 다 "숫자를 계산합니다." → LLM이 둘을 구분할 단서가 없음
- **명확 버전**: 이름 `calc_c`(합), `calc_d`(곱), docstring에 **무엇을 계산하는지** 명시
- 등록 순서를 교차해 위치 편향도 함께 봅니다.

In [55]:
def evaluate(label: str, sum_tool, mul_tool):
    """두 도구를 두 가지 등록 순서로 등록하고, 질문별로 올바른 도구를 골랐는지 채점"""
    expected = {"sum": sum_tool.name, "mul": mul_tool.name}
    orders = {"[합, 곱]": [sum_tool, mul_tool], "[곱, 합]": [mul_tool, sum_tool]}
    for order_label, tools in orders.items():
        hits = 0
        for q, kind in CASES:
            got = called_tools(q, tools)
            ok = got == [expected[kind]]
            hits += ok
            print(f"  {'✅' if ok else '❌'} {q} → {got or '호출 안 함'} (정답: {expected[kind]})")
        print(f"  ▶ {label} · 등록 순서 {order_label}: {hits}/{len(CASES)} 정답\n")

print("✅ 모호/명확 도구 + 채점 함수 준비 완료")

✅ 모호/명확 도구 + 채점 함수 준비 완료


In [56]:
# 모호 버전: 기능은 다르지만 docstring이 똑같이 모호
@tool
def calc_a(a: int, b: int) -> int:
    """숫자를 계산합니다."""
    return a + b          # 실제 기능: 합

@tool
def calc_b(a: int, b: int) -> int:
    """숫자를 계산합니다."""
    return a * b          # 실제 기능: 곱

# 명확 버전: 같은 기능, docstring에 무엇을 계산하는지 명시
@tool
def calc_c(a: int, b: int) -> int:
    """두 정수의 합(덧셈)을 계산합니다."""
    return a + b

@tool
def calc_d(a: int, b: int) -> int:
    """두 정수의 곱(곱셈)을 계산합니다."""
    return a * b

# (질문, 정답 종류) — 질문에 연산이 드러나 있으므로 정답이 명확함
CASES = [
    ("3 더하기 5는?", "sum"),
    ("12 더하기 30은?", "sum"),
    ("3 곱하기 5는?", "mul"),
    ("7 곱하기 8은?", "mul"),
]

In [58]:
# 모호한 docstring: calc_a(합) vs calc_b(곱) — 이름도 docstring도 구분 단서 없음
print("🔸 모호 버전 (calc_a=합, calc_b=곱)\n")
evaluate("모호", calc_a, calc_b)

# 명확한 docstring: calc_c(합) vs calc_d(곱) — 같은 질문, 같은 순서 교차
print("🔹 명확 버전 (calc_c=합, calc_d=곱)\n")
evaluate("명확", calc_c, calc_d)

🔸 모호 버전 (calc_a=합, calc_b=곱)

  ✅ 3 더하기 5는? → ['calc_a'] (정답: calc_a)
  ✅ 12 더하기 30은? → ['calc_a'] (정답: calc_a)
  ❌ 3 곱하기 5는? → ['calc_a'] (정답: calc_b)
  ❌ 7 곱하기 8은? → ['calc_a'] (정답: calc_b)
  ▶ 모호 · 등록 순서 [합, 곱]: 2/4 정답

  ✅ 3 더하기 5는? → ['calc_a'] (정답: calc_a)
  ✅ 12 더하기 30은? → ['calc_a'] (정답: calc_a)
  ❌ 3 곱하기 5는? → ['calc_a'] (정답: calc_b)
  ❌ 7 곱하기 8은? → ['calc_a'] (정답: calc_b)
  ▶ 모호 · 등록 순서 [곱, 합]: 2/4 정답

🔹 명확 버전 (calc_c=합, calc_d=곱)

  ✅ 3 더하기 5는? → ['calc_c'] (정답: calc_c)
  ✅ 12 더하기 30은? → ['calc_c'] (정답: calc_c)
  ✅ 3 곱하기 5는? → ['calc_d'] (정답: calc_d)
  ✅ 7 곱하기 8은? → ['calc_d'] (정답: calc_d)
  ▶ 명확 · 등록 순서 [합, 곱]: 4/4 정답

  ✅ 3 더하기 5는? → ['calc_c'] (정답: calc_c)
  ✅ 12 더하기 30은? → ['calc_c'] (정답: calc_c)
  ✅ 3 곱하기 5는? → ['calc_d'] (정답: calc_d)
  ✅ 7 곱하기 8은? → ['calc_d'] (정답: calc_d)
  ▶ 명확 · 등록 순서 [곱, 합]: 4/4 정답



## D-7. 실험 ⑤ docstring은 모호, 함수명은 명확

D-6의 두 버전 사이를 채우는 조합입니다.

| 버전 | 함수명 | docstring |
| --- | --- | --- |
| 모호 (D-6) | `calc_a`, `calc_b` | 모호 |
| **이름만 명확 (이번 실험)** | `add_numbers`, `multiply_numbers` | **모호** |
| 명확 (D-6) | `calc_c`, `calc_d` | 명확 |

→ **함수명만으로도 LLM이 도구를 구분할 수 있는지** 확인합니다. 채점 함수 `evaluate`는 D-6의 것을 그대로 씁니다.

In [54]:
# 이름은 명확, docstring은 D-6 모호 버전과 동일
@tool
def add_numbers(a: int, b: int) -> int:
    """숫자를 계산합니다."""
    return a + b

@tool
def multiply_numbers(a: int, b: int) -> int:
    """숫자를 계산합니다."""
    return a * b

print("🔸 이름만 명확 버전 (add_numbers=합, multiply_numbers=곱)\n")
evaluate("이름만 명확", add_numbers, multiply_numbers)

🔸 이름만 명확 버전 (add_numbers=합, multiply_numbers=곱)

  ✅ 3 더하기 5는? → ['add_numbers'] (정답: add_numbers)
  ✅ 12 더하기 30은? → ['add_numbers'] (정답: add_numbers)
  ✅ 3 곱하기 5는? → ['multiply_numbers'] (정답: multiply_numbers)
  ✅ 7 곱하기 8은? → ['multiply_numbers'] (정답: multiply_numbers)
  ▶ 이름만 명확 · 등록 순서 [합, 곱]: 4/4 정답

  ✅ 3 더하기 5는? → ['add_numbers'] (정답: add_numbers)
  ✅ 12 더하기 30은? → ['add_numbers'] (정답: add_numbers)
  ✅ 3 곱하기 5는? → ['multiply_numbers'] (정답: multiply_numbers)
  ✅ 7 곱하기 8은? → ['multiply_numbers'] (정답: multiply_numbers)
  ▶ 이름만 명확 · 등록 순서 [곱, 합]: 4/4 정답

